# create_schema.ipynb
## Drilling Operations — SQLite Database Schema Creation

### Design Summary
| Table | Primary Key | Type | Notes |
|---|---|---|---|
| PAC | PacName | Natural | Top-level organisational entity |
| Region | RegionName | Natural | Belongs to one PAC |
| Field | FieldName | Natural | Belongs to one Region |
| Rig | RigName | Natural | Independent reference entity |
| Well | WellName | Natural | Central hub; references Field & Rig |
| AFE | IDAFE | Surrogate | Multiple AFEs per well possible |
| WellNpt | IDWellNpt | Surrogate | Multiple NPT records per well |
| WellCompletionCost | IDWellCompletionCost | Surrogate | Multiple completion cost records per well |
| WellDrilling | IDWellDrilling | Surrogate | Multiple drilling records per well |
| Report | IDReport | Surrogate | Multiple reports per well |

> **Surrogate keys** are used where no natural
> unique identifier exists in the source flat table. SQLite's `INTEGER PRIMARY KEY`
> auto-increments these automatically.

In [43]:
import sqlite3
import os

# ---------------------------------------------------------------------------
# Database path — placed in the same directory as this notebook
# ---------------------------------------------------------------------------
DB_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'drilling_operations.db')

# Drop and recreate for a clean run
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Enable foreign key enforcement (SQLite disables it by default)
cursor.execute('PRAGMA foreign_keys = ON;')

print(f'Connected to: {DB_PATH}')

Connected to: c:\Users\AEM-Mior\OneDrive - Aem Energy Solutions\Working File\5. PROJECT\2026\13-PYTHON FUNDAMENTAL\Python-Submission\Data-Journey-Kickstart-Python-Assessment\Mior_DataEngineering_Assessment\drilling_operations.db


## 1. PAC Table
Top-level organisational entity.  
`PacName` is a natural primary key — it is unique and stable in the source data.

In [44]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS PAC (
        PacName  TEXT  NOT NULL,

        CONSTRAINT pk_pac PRIMARY KEY (PacName)
    );
''')

print('PAC table created.')

PAC table created.


## 2. Region Table
`RegionName` is a natural primary key.  
`PacName` is a foreign key — each region belongs to exactly one PAC (one-to-many).

In [45]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Region (
        RegionName  TEXT  NOT NULL,
        PacName     TEXT,

        CONSTRAINT pk_region  PRIMARY KEY (RegionName),
        CONSTRAINT fk_region_pac FOREIGN KEY (PacName)
            REFERENCES PAC (PacName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('Region table created.')

Region table created.


## 3. Field Table
`FieldName` is a natural primary key.  
`RegionName` is a foreign key — each field belongs to exactly one region (one-to-many).

In [46]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Field (
        FieldName   TEXT  NOT NULL,
        RegionName  TEXT,

        CONSTRAINT pk_field  PRIMARY KEY (FieldName),
        CONSTRAINT fk_field_region FOREIGN KEY (RegionName)
            REFERENCES Region (RegionName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('Field table created.')

Field table created.


## 4. Rig Table
`RigName` is a natural primary key.  
Rig is an **independent reference entity** — a rig can be assigned to wells across
different fields and regions, so it is not part of the main hierarchy.

In [47]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Rig (
        RigName  TEXT  NOT NULL,
        RigType  TEXT,

        CONSTRAINT pk_rig PRIMARY KEY (RigName)
    );
''')

print('Rig table created.')

Rig table created.


## 5. Well Table
`WellName` is a natural primary key.  
The Well table is the **central hub** of the schema — all child detail tables
(AFE, WellNpt, WellCompletionCost, WellDrilling, Report) reference `WellName`.

- `SpudDate` — stored as `DATE` (date only, no time component)
- `WellStartDateTime` / `WellEndDateTime` — stored as `DATETIME` (full timestamp)

In [48]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Well (
        WellName          TEXT     NOT NULL,
        FieldName         TEXT,
        RigName           TEXT,
        WellType          TEXT,
        WaterDepth        REAL,         -- Water depth in feet/metres
        Year              INTEGER,      -- Year the well was drilled
        SpudDate          DATE,         -- Date-only field
        WellStartDateTime DATETIME,     -- Full timestamps
        WellEndDateTime   DATETIME,     -- Full timestamps

        CONSTRAINT pk_well PRIMARY KEY (WellName),
        CONSTRAINT fk_well_field FOREIGN KEY (FieldName)
            REFERENCES Field (FieldName)
            ON UPDATE CASCADE
            ON DELETE SET NULL,
        CONSTRAINT fk_well_rig FOREIGN KEY (RigName)
            REFERENCES Rig (RigName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('Well table created.')

Well table created.


## 6. AFE Table
`IDAFE` is a **surrogate primary key**  
A surrogate key is required because no natural unique identifier exists for AFE records
in the source flat table, and multiple AFE records can exist per well.

SQLite's `INTEGER PRIMARY KEY` auto-increments — no manual ID assignment needed.

In [49]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS AFE (
        IDAFE      INTEGER  NOT NULL,
        WellName   TEXT,
        AfeCost    REAL,    
        AfeDays    REAL,    
        FinalCost  REAL,    
        FinalDays  REAL,    

        CONSTRAINT pk_afe PRIMARY KEY (IDAFE AUTOINCREMENT),
        CONSTRAINT fk_afe_well FOREIGN KEY (WellName)
            REFERENCES Well (WellName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('AFE table created.')

AFE table created.


## 7. WellNpt Table
`IDWellNpt` is a **surrogate primary key**  
NPT (Non-Productive Time) — multiple readings can exist per well, and the source
data provides no unique identifier per NPT row.

In [50]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS WellNpt (
        IDWellNpt             INTEGER  NOT NULL,
        WellName              TEXT,
        WellNptPercentageWow  REAL,    
        WellNptPercentage     REAL,   

        CONSTRAINT pk_wellnpt PRIMARY KEY (IDWellNpt AUTOINCREMENT),
        CONSTRAINT fk_wellnpt_well FOREIGN KEY (WellName)
            REFERENCES Well (WellName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('WellNpt table created.')

WellNpt table created.


## 8. WellCompletionCost Table
`IDWellCompletionCost` is a **surrogate primary key**  
Completion cost data is separated from the Well table to isolate post-drilling financial
data and to support multiple completion cost records per well.

In [51]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS WellCompletionCost (
        IDWellCompletionCost  INTEGER  NOT NULL,
        WellName              TEXT,
        CompletionCostPlan    REAL,    
        CompletionCostActual  REAL,    

        CONSTRAINT pk_wellcompletioncost PRIMARY KEY (IDWellCompletionCost AUTOINCREMENT),
        CONSTRAINT fk_wellcompletioncost_well FOREIGN KEY (WellName)
            REFERENCES Well (WellName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('WellCompletionCost table created.')

WellCompletionCost table created.


## 9. WellDrilling Table
`IDWellDrilling` is a **surrogate primary key**  
Drilling performance metrics (Wcpf = Well Cost Per Foot) are isolated into their own
table to keep Well focused on core metadata and support multiple performance records.

In [52]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS WellDrilling (
        IDWellDrilling    INTEGER  NOT NULL,
        WellName          TEXT,
        DrillingPlanWcpf    REAL,   
        DrillingActualWcpf  REAL, 

        CONSTRAINT pk_welldrilling PRIMARY KEY (IDWellDrilling AUTOINCREMENT),
        CONSTRAINT fk_welldrilling_well FOREIGN KEY (WellName)
            REFERENCES Well (WellName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('WellDrilling table created.')

WellDrilling table created.


## 10. Report Table
`IDReport` is a **surrogate primary key**
One well can have many associated reports across different submission
dates. The source flat table provides no unique identifier per report row.

- `DocumentDate` / `SubmittedAt` — stored as `DATETIME`; timezone offsets (+0000, +0800)
  are normalised to UTC-naive on ingestion.

In [53]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Report (
        IDReport      INTEGER  NOT NULL,
        WellName      TEXT,
        ReportType    TEXT,      
        DocumentName  TEXT,      
        DocumentDate  DATETIME,  
        SubmittedAt   DATETIME,  
        SubmittedBy   TEXT,      

        CONSTRAINT pk_report PRIMARY KEY (IDReport AUTOINCREMENT),
        CONSTRAINT fk_report_well FOREIGN KEY (WellName)
            REFERENCES Well (WellName)
            ON UPDATE CASCADE
            ON DELETE SET NULL
    );
''')

print('Report table created.')

Report table created.


## Commit & Verify
Commit all DDL statements and verify the tables were created successfully.

In [54]:
conn.commit()

# ---------------------------------------------------------------------------
# Verify: list all tables in the database
# ---------------------------------------------------------------------------
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()

print(f'\nDatabase: {DB_PATH}')
print(f'Tables created ({len(tables)}):')
for t in tables:
    print(f'  - {t[0]}')


Database: c:\Users\AEM-Mior\OneDrive - Aem Energy Solutions\Working File\5. PROJECT\2026\13-PYTHON FUNDAMENTAL\Python-Submission\Data-Journey-Kickstart-Python-Assessment\Mior_DataEngineering_Assessment\drilling_operations.db
Tables created (11):
  - AFE
  - Field
  - PAC
  - Region
  - Report
  - Rig
  - Well
  - WellCompletionCost
  - WellDrilling
  - WellNpt
  - sqlite_sequence


In [55]:
# ---------------------------------------------------------------------------
# Verify: show column definitions for each table
# ---------------------------------------------------------------------------
for (table_name,) in tables:
    print(f'\n--- {table_name} ---')
    cursor.execute(f'PRAGMA table_info({table_name});')
    cols = cursor.fetchall()
    print(f'  {"cid":<4} {"name":<25} {"type":<12} {"notnull":<8} {"pk"}')
    print(f'  {"-"*4} {"-"*25} {"-"*12} {"-"*8} {"-"*4}')
    for col in cols:
        cid, name, col_type, notnull, dflt, pk = col
        print(f'  {cid:<4} {name:<25} {col_type:<12} {str(bool(notnull)):<8} {pk}')


--- AFE ---
  cid  name                      type         notnull  pk
  ---- ------------------------- ------------ -------- ----
  0    IDAFE                     INTEGER      True     1
  1    WellName                  TEXT         False    0
  2    AfeCost                   REAL         False    0
  3    AfeDays                   REAL         False    0
  4    FinalCost                 REAL         False    0
  5    FinalDays                 REAL         False    0

--- Field ---
  cid  name                      type         notnull  pk
  ---- ------------------------- ------------ -------- ----
  0    FieldName                 TEXT         True     1
  1    RegionName                TEXT         False    0

--- PAC ---
  cid  name                      type         notnull  pk
  ---- ------------------------- ------------ -------- ----
  0    PacName                   TEXT         True     1

--- Region ---
  cid  name                      type         notnull  pk
  ---- -----------

In [56]:
# ---------------------------------------------------------------------------
# Verify: show foreign key definitions for each table
# ---------------------------------------------------------------------------
print('Foreign Key Relationships:\n')
for (table_name,) in tables:
    cursor.execute(f'PRAGMA foreign_key_list({table_name});')
    fks = cursor.fetchall()
    if fks:
        for fk in fks:
            print(f'  {table_name}.{fk[3]}  -->  {fk[2]}.{fk[4]}')

conn.close()
print('\nConnection closed. Schema creation complete.')

Foreign Key Relationships:

  AFE.WellName  -->  Well.WellName
  Field.RegionName  -->  Region.RegionName
  Region.PacName  -->  PAC.PacName
  Report.WellName  -->  Well.WellName
  Well.RigName  -->  Rig.RigName
  Well.FieldName  -->  Field.FieldName
  WellCompletionCost.WellName  -->  Well.WellName
  WellDrilling.WellName  -->  Well.WellName
  WellNpt.WellName  -->  Well.WellName

Connection closed. Schema creation complete.
